In [0]:
# ============================================================
# FOOD DELIVERY PERFORMANCE & CUSTOMER ANALYTICS PIPELINE
# Silver Layer - Data Cleaning and Transformation
# Roll Number: 23052151
# ============================================================

print("Food Delivery Analytics Pipeline")
print("Roll Number: 23052151")
print("Layer: Silver")

Food Delivery Analytics Pipeline
Roll Number: 23052151
Layer: Silver


In [0]:
# ============================================================
# PATHS
# ============================================================

BRONZE_PATH = "/Volumes/workspace/capstone_23052151/bronze_data"
SILVER_PATH = "/Volumes/workspace/capstone_23052151/silver_data"

print("Bronze path:", BRONZE_PATH)
print("Silver path:", SILVER_PATH)

Bronze path: /Volumes/workspace/capstone_23052151/bronze_data
Silver path: /Volumes/workspace/capstone_23052151/silver_data


In [0]:
# ============================================================
# READ BRONZE DATA
# ============================================================

customers_df = spark.read.format("delta").load(
    f"{BRONZE_PATH}/customers"
)

restaurants_df = spark.read.format("delta").load(
    f"{BRONZE_PATH}/restaurants"
)

menu_items_df = spark.read.format("delta").load(
    f"{BRONZE_PATH}/menu_items"
)

orders_df = spark.read.format("delta").load(
    f"{BRONZE_PATH}/orders"
)

order_items_df = spark.read.format("delta").load(
    f"{BRONZE_PATH}/order_items"
)

delivery_details_df = spark.read.format("delta").load(
    f"{BRONZE_PATH}/delivery_details"
)

reviews_df = spark.read.format("delta").load(
    f"{BRONZE_PATH}/reviews"
)

print("All Bronze datasets loaded successfully.")

All Bronze datasets loaded successfully.


In [0]:
# ============================================================
# CLEAN CUSTOMERS
# ============================================================

from pyspark.sql.functions import (
    col,
    trim,
    to_date,
    when
)

customers_silver = (
    customers_df
    .withColumn("customer_id", col("customer_id").cast("string"))
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("city", trim(col("city")))
    .withColumn("customer_segment", trim(col("customer_segment")))
    .withColumn("signup_date", to_date(col("signup_date")))
    .dropDuplicates(["customer_id"])
)

print("Customers Silver rows:", customers_silver.count())

Customers Silver rows: 20000


In [0]:
# ============================================================
# CLEAN RESTAURANTS
# ============================================================

restaurants_silver = (
    restaurants_df
    .withColumn("restaurant_id", col("restaurant_id").cast("string"))
    .withColumn("restaurant_name", trim(col("restaurant_name")))
    .withColumn("city", trim(col("city")))
    .withColumn("cuisine_type", trim(col("cuisine_type")))
    .withColumn(
        "restaurant_rating",
        col("restaurant_rating").cast("double")
    )
    .dropDuplicates(["restaurant_id"])
)

print("Restaurants Silver rows:", restaurants_silver.count())

Restaurants Silver rows: 500


In [0]:
# ============================================================
# CLEAN MENU ITEMS
# ============================================================

menu_items_silver = (
    menu_items_df
    .withColumn("item_id", col("item_id").cast("string"))
    .withColumn("restaurant_id", col("restaurant_id").cast("string"))
    .withColumn("item_name", trim(col("item_name")))
    .withColumn("category", trim(col("category")))
    .withColumn("unit_price", col("unit_price").cast("double"))
    .dropDuplicates(["item_id"])
)

print("Menu Items Silver rows:", menu_items_silver.count())

Menu Items Silver rows: 5000


In [0]:
# ============================================================
# CLEAN ORDERS
# ============================================================

orders_silver = (
    orders_df
    .withColumn("order_id", col("order_id").cast("string"))
    .withColumn("customer_id", col("customer_id").cast("string"))
    .withColumn("restaurant_id", col("restaurant_id").cast("string"))
    .withColumn("order_datetime", col("order_datetime").cast("timestamp"))
    .withColumn("order_status", trim(col("order_status")))
    .withColumn("payment_method", trim(col("payment_method")))
    .withColumn(
        "discount_amount",
        col("discount_amount").cast("double")
    )
    .dropDuplicates(["order_id"])
)

print("Raw Orders:", orders_df.count())
print("Silver Orders:", orders_silver.count())
print(
    "Duplicate Orders Removed:",
    orders_df.count() - orders_silver.count()
)

Raw Orders: 102000
Silver Orders: 100000
Duplicate Orders Removed: 2000


In [0]:
# ============================================================
# CLEAN ORDER ITEMS
# ============================================================

from pyspark.sql.functions import col, trim, expr

order_items_clean = (
    order_items_df
    .withColumn(
        "order_id",
        col("order_id").cast("string")
    )
    .withColumn(
        "item_id",
        col("item_id").cast("string")
    )
    .withColumn(
        "category",
        trim(col("category"))
    )
    .withColumn(
        "quantity",
        expr("try_cast(quantity AS INT)")
    )
    .withColumn(
        "unit_price",
        expr("try_cast(unit_price AS DOUBLE)")
    )
    .withColumn(
        "item_revenue",
        expr("try_cast(item_revenue AS DOUBLE)")
    )
)

print("Order items cleaned successfully.")

Order items cleaned successfully.


In [0]:
# Keep only order items belonging to valid orders

valid_order_items = (
    order_items_clean
    .join(
        orders_silver.select("order_id"),
        on="order_id",
        how="inner"
    )
)

print("Raw Order Items:", order_items_df.count())
print("Silver Order Items:", valid_order_items.count())
print(
    "Orphan Order Items Removed:",
    order_items_df.count() - valid_order_items.count()
)

Raw Order Items: 165248
Silver Order Items: 165098
Orphan Order Items Removed: 150


In [0]:
# ============================================================
# CLEAN DELIVERY DETAILS
# ============================================================

delivery_silver = (
    delivery_details_df
    .withColumn("order_id", col("order_id").cast("string"))
    .withColumn(
        "estimated_delivery_time",
        col("estimated_delivery_time").cast("timestamp")
    )
    .withColumn(
        "actual_delivery_time",
        col("actual_delivery_time").cast("timestamp")
    )
    .withColumn(
        "delivery_partner",
        trim(col("delivery_partner"))
    )
    .withColumn(
        "delivery_distance_km",
        col("delivery_distance_km").cast("double")
    )
    .dropDuplicates(["order_id"])
)

print("Delivery Silver rows:", delivery_silver.count())

Delivery Silver rows: 100000


In [0]:
# ============================================================
# CLEAN REVIEWS
# ============================================================

reviews_silver = (
    reviews_df
    .withColumn("review_id", col("review_id").cast("string"))
    .withColumn("order_id", col("order_id").cast("string"))
    .withColumn("rating", col("rating").cast("double"))
    .withColumn("review_text", trim(col("review_text")))
    .withColumn("review_date", to_date(col("review_date")))
    .dropDuplicates(["review_id"])
)

print("Reviews Silver rows:", reviews_silver.count())

Reviews Silver rows: 80000


In [0]:
# ============================================================
# DELIVERY PERFORMANCE
# ============================================================

from pyspark.sql.functions import (
    when,
    unix_timestamp
)

delivery_performance = (
    orders_silver
    .join(
        restaurants_silver.select(
            "restaurant_id",
            "restaurant_name",
            "city",
            "cuisine_type"
        ),
        on="restaurant_id",
        how="left"
    )
    .join(
        delivery_silver,
        on="order_id",
        how="left"
    )
    .withColumn(
        "delivery_delay_minutes",
        (
            unix_timestamp("actual_delivery_time")
            - unix_timestamp("estimated_delivery_time")
        ) / 60
    )
    .withColumn(
        "is_late",
        when(
            col("delivery_delay_minutes") > 0,
            1
        ).otherwise(0)
    )
)

display(delivery_performance.limit(20))

order_id,restaurant_id,customer_id,order_datetime,order_status,payment_method,discount_amount,restaurant_name,city,cuisine_type,estimated_delivery_time,actual_delivery_time,delivery_partner,delivery_distance_km,delivery_delay_minutes,is_late
O0000039,R0146,C009635,2024-06-23T05:29:35.000Z,DELIVERED,WALLET,20.0,Restaurant_0146,Chennai,Fast Food,2024-06-23T05:58:35.000Z,2024-06-23T06:01:35.000Z,Partner_B,13.86,3.0,1
O0000053,R0332,C012006,2024-03-19T22:48:52.000Z,DELIVERED,WALLET,75.0,Restaurant_0332,Delhi,Bengali,2024-03-19T23:47:52.000Z,2024-03-19T23:43:52.000Z,Partner_A,5.36,-4.0,0
O0000062,R0392,C008177,2024-01-13T06:39:40.000Z,DELIVERED,UPI,0.0,Restaurant_0392,Bengaluru,Indian,2024-01-13T07:19:40.000Z,2024-01-13T07:26:40.000Z,Partner_B,11.35,7.0,1
O0000074,R0240,C001337,2024-02-21T08:34:48.000Z,DELIVERED,CARD,0.0,Restaurant_0240,Mumbai,Fast Food,2024-02-21T09:31:48.000Z,2024-02-21T10:01:48.000Z,Partner_C,2.73,30.0,1
O0000104,R0416,C017138,2024-04-17T04:43:49.000Z,DELIVERED,CARD,0.0,Restaurant_0416,Mumbai,Indian,2024-04-17T05:17:49.000Z,2024-04-17T05:26:49.000Z,Partner_A,12.34,9.0,1
O0000129,R0155,C004769,2024-04-18T13:21:10.000Z,DELIVERED,CARD,0.0,Restaurant_0155,Mumbai,Continental,2024-04-18T14:08:10.000Z,2024-04-18T14:25:10.000Z,Partner_D,12.8,17.0,1
O0000139,R0121,C010906,2024-04-06T14:37:04.000Z,DELIVERED,CARD,30.0,Restaurant_0121,Kolkata,South Indian,2024-04-06T15:34:04.000Z,2024-04-06T15:49:04.000Z,Partner_C,9.62,15.0,1
O0000211,R0226,C017065,2024-11-06T01:44:25.000Z,DELIVERED,WALLET,0.0,Restaurant_0226,Delhi,Bengali,2024-11-06T02:13:25.000Z,2024-11-06T02:12:25.000Z,Partner_B,1.4,-1.0,0
O0000212,R0050,C007548,2024-06-10T08:42:21.000Z,DELIVERED,CARD,50.0,Restaurant_0050,Mumbai,Indian,2024-06-10T09:37:21.000Z,2024-06-10T09:34:21.000Z,Partner_B,13.53,-3.0,0
O0000231,R0240,C010370,2024-11-27T05:41:20.000Z,DELIVERED,UPI,30.0,Restaurant_0240,Mumbai,Fast Food,2024-11-27T06:11:20.000Z,2024-11-27T06:27:20.000Z,Partner_A,5.92,16.0,1


In [0]:
# ============================================================
# SILVER QUALITY CHECKS
# ============================================================

print("========== SILVER QUALITY CHECKS ==========")

print("Customers:", customers_silver.count())
print("Restaurants:", restaurants_silver.count())
print("Menu Items:", menu_items_silver.count())
print("Orders:", orders_silver.count())
print("Order Items:", valid_order_items.count())
print("Delivery Details:", delivery_silver.count())
print("Reviews:", reviews_silver.count())

print("\nNull order IDs in Orders:")
print(
    orders_silver.filter(
        col("order_id").isNull()
    ).count()
)

print("\nNull customer IDs in Orders:")
print(
    orders_silver.filter(
        col("customer_id").isNull()
    ).count()
)

print("\nNull restaurant IDs in Orders:")
print(
    orders_silver.filter(
        col("restaurant_id").isNull()
    ).count()
)

========== SILVER QUALITY CHECKS ==========
Customers: 20000
Restaurants: 500
Menu Items: 5000
Orders: 100000
Order Items: 165098
Delivery Details: 100000
Reviews: 80000

Null order IDs in Orders:
0

Null customer IDs in Orders:
0

Null restaurant IDs in Orders:
0


In [0]:
# ============================================================
# WRITE SILVER DATA
# ============================================================

customers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/customers")

restaurants_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/restaurants")

menu_items_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/menu_items")

orders_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/orders")

valid_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/order_items")

delivery_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/delivery_details")

reviews_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/reviews")

delivery_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/delivery_performance")

print("Silver layer written successfully.")

Silver layer written successfully.


In [0]:
# ============================================================
# VERIFY SILVER LAYER
# ============================================================

display(dbutils.fs.ls(SILVER_PATH))

path,name,size,modificationTime
dbfs:/Volumes/workspace/capstone_23052151/silver_data/customers/,customers/,0,1790657909591
dbfs:/Volumes/workspace/capstone_23052151/silver_data/delivery_details/,delivery_details/,0,1790657909591
dbfs:/Volumes/workspace/capstone_23052151/silver_data/delivery_performance/,delivery_performance/,0,1790657909591
dbfs:/Volumes/workspace/capstone_23052151/silver_data/menu_items/,menu_items/,0,1790657909591
dbfs:/Volumes/workspace/capstone_23052151/silver_data/order_items/,order_items/,0,1790657909591
dbfs:/Volumes/workspace/capstone_23052151/silver_data/orders/,orders/,0,1790657909591
dbfs:/Volumes/workspace/capstone_23052151/silver_data/restaurants/,restaurants/,0,1790657909591
dbfs:/Volumes/workspace/capstone_23052151/silver_data/reviews/,reviews/,0,1790657909591
